# WaveGuard Colab — yolo26s 파인튜닝 (Python API만 사용)

이 노트북은 `!yolo` / `python -m ultralytics` 를 **쓰지 않습니다**.
Colab에서 CLI가 깨지는 문제를 피하기 위해 `from ultralytics import YOLO` 만 사용합니다.

## 준비
1. 로컬에서 `finetune/make_dataset.py` 로 만든 `gwangalli_dataset.zip` 또는 `gwangalli_colab.zip` 을 Drive에 업로드
2. 아래 **GPU(H100/Premium) 체크리스트**대로 런타임 연결
3. **1) GPU 확인** 셀 실행 → 이름·VRAM 확인 후 학습 진행
4. 아래 셀을 **위에서부터** 실행
5. (선택) **4.5절**에서 의심 프레임 목록 확인 후 `PURGE_ALL = True` 로 전부 삭제

학습 후 `best.pt` → 로컬 `vision/models/yolo26s_beach_ft.pt` 에 넣고 서버 재시작.

---

## GPU(H100 / Premium) 체크리스트 — Pro+

> H100은 **보장되지 않습니다**. CU·물량에 따라 A100/L4/T4가 올 수 있습니다.  
> WaveGuard 파인튜닝은 **A100이면 충분**합니다. H100만 고집할 필요는 없습니다.

### A. 결제·잔량
1. [Colab Pro+](https://colab.research.google.com/signup) 가입 (Premium GPU 접근에 유리)
2. 노트북 우측 상단 **연결 ▾ → 리소스 보기(View resources)**
3. **Compute Units(CU)** 잔량 확인 — 0이면 Premium 연결 실패/Free급으로 떨어짐
4. 부족하면 **Pay as you go**로 CU 추가 구매

### B. 런타임 클릭 순서
1. 메뉴 **런타임 → 런타임 유형 변경**
2. **하드웨어 가속기** = `GPU`
3. GPU 종류가 보이면 **프리미엄(Premium)** / 고성능 옵션 선택  
   (UI 문구는 `Premium GPU`, `A100`, `H100` 등으로 시기에 따라 다름)
4. (가능하면) **고용량 RAM** 도 함께 선택
5. **저장** → 우측 상단 **연결** (이미 연결돼 있으면 **런타임 → 세션 다시 시작** 후 재연결)

### C. 안 잡힐 때
- 런타임 연결 해제 → 1~2분 대기 → B 다시
- 다른 시간대에 재시도 (퇴근 후·주말 혼잡)
- `nvidia-smi`에 T4만 나오면: Premium 미할당 → CU/물량 문제
- **반드시 H100**이면 Colab 대신 RunPod 등 지정 대여가 더 빠름

### D. 학습 전 확인 (아래 1번 셀)
- `H100` / `A100` / `L4` / `T4` 중 무엇인지
- VRAM(GiB), CUDA 사용 가능 여부
- CU가 빨리 닳으니 학습 끝나면 **런타임 → 런타임 연결 해제**

In [ ]:
# 1) GPU 확인 — Premium/H100 할당 여부 체크리스트용
# 기대: NVIDIA H100 / A100 (또는 L4). T4면 Premium이 안 붙은 것.
!nvidia-smi

import torch

ok = torch.cuda.is_available()
name = torch.cuda.get_device_name(0) if ok else "CPU"
vram_gb = (
    round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
    if ok else 0
)
print("=" * 50)
print("CUDA          :", ok)
print("GPU name      :", name)
print("VRAM (approx) :", vram_gb, "GiB")
print("=" * 50)

upper = name.upper()
if "H100" in upper:
    print("OK: H100 할당됨 — 학습 진행해도 됩니다.")
elif "A100" in upper:
    print("OK: A100 할당됨 — WaveGuard 파인튜닝에 충분합니다.")
elif "L4" in upper or "V100" in upper:
    print("주의: 중급 GPU — 학습 가능, batch/imgsz를 조금 낮추세요.")
elif "T4" in upper:
    print("경고: T4 — Premium/고성능 GPU가 아닙니다.")
    print("  → 런타임 유형에서 Premium GPU 재선택, CU 잔량 확인 후 재연결")
elif not ok:
    print("실패: GPU 없음 — 런타임 → 런타임 유형 변경 → GPU 선택")
else:
    print("정보: 위 GPU name을 확인하세요:", name)


In [ ]:
# 2) ultralytics 설치 + API 확인 (CLI 사용 안 함)
!pip -q install -U ultralytics
import torch
from ultralytics import YOLO
import ultralytics
ultralytics.checks()
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — 런타임을 GPU로 바꾸세요')
print('YOLO API OK')

In [ ]:
# 3) Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4) 데이터셋 준비
# 로컬 PC 파일:
#   vision/finetune/gwangalli_dataset.zip  또는  gwangalli_colab.zip
# → Google Drive에 올린 뒤 이 셀 실행. 없으면 자동으로 업로드 창을 띄움.
import os, zipfile, shutil, glob

CANDIDATES = [
    '/content/drive/MyDrive/gwangalli/gwangalli_dataset.zip',
    '/content/drive/MyDrive/gwangalli/gwangalli_colab.zip',
    '/content/drive/MyDrive/gwangalli_dataset.zip',
    '/content/drive/MyDrive/gwangalli_colab.zip',
    '/content/drive/MyDrive/haeundae-crowd-density/vision/finetune/gwangalli_dataset.zip',
    '/content/drive/MyDrive/haeundae-crowd-density/vision/finetune/gwangalli_colab.zip',
]

# Drive에서 이름으로도 검색
found = [p for p in CANDIDATES if os.path.exists(p)]
if not found:
    found = glob.glob('/content/drive/MyDrive/**/gwangalli_dataset.zip', recursive=True)
if not found:
    found = glob.glob('/content/drive/MyDrive/**/gwangalli_colab.zip', recursive=True)

if found:
    ZIP = found[0]
    print('찾은 zip:', ZIP)
else:
    print('Drive에서 zip을 못 찾았습니다.')
    print('로컬의 vision/finetune/gwangalli_dataset.zip 을 선택해 업로드하세요.')
    from google.colab import files
    up = files.upload()  # 파일 선택 창
    assert up, '업로드가 취소되었습니다'
    name = next(iter(up))
    ZIP = '/content/' + name
    open(ZIP, 'wb').write(up[name])
    print('업로드 완료:', ZIP, len(up[name]) // (1024 * 1024), 'MB')

# 압축 해제
if os.path.basename(ZIP).endswith('gwangalli_colab.zip') or 'gwangalli_colab' in ZIP:
    if os.path.exists('/content/pkg'):
        shutil.rmtree('/content/pkg')
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content/pkg')
    # zip 내부 구조가 vision/... 또는 pkg/vision/... 일 수 있음
    for cand in (
        '/content/pkg/vision/finetune/dataset',
        '/content/pkg/finetune/dataset',
    ):
        if os.path.isdir(cand):
            ROOT = cand
            break
    else:
        raise FileNotFoundError('gwangalli_colab.zip 안에 finetune/dataset 이 없습니다')
    DATA = os.path.join(ROOT, 'data.yaml')
    open(DATA, 'w').write(
        f'path: {ROOT}\ntrain: train/images\nval: val/images\n'
        'names:\n  0: person\n  1: tube\n'
    )
else:
    DST = '/content/gwangalli'
    if os.path.exists(DST):
        shutil.rmtree(DST)
    os.makedirs(DST, exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DST)
    # data.yaml 이 루트 또는 하위일 수 있음
    DATA = os.path.join(DST, 'data.yaml')
    if not os.path.exists(DATA):
        hits = glob.glob(DST + '/**/data.yaml', recursive=True)
        assert hits, 'zip 안에 data.yaml 없음'
        DATA = hits[0]
        DST = os.path.dirname(DATA)
    txt = open(DATA, encoding='utf-8').read()
    if 'path: .' in txt:
        txt = txt.replace('path: .', f'path: {DST}')
    else:
        # path 줄을 절대경로로 덮어쓰기
        lines = []
        for line in txt.splitlines():
            if line.startswith('path:'):
                lines.append(f'path: {DST}')
            else:
                lines.append(line)
        txt = '\n'.join(lines) + '\n'
    open(DATA, 'w', encoding='utf-8').write(txt)

print('data.yaml:\n', open(DATA, encoding='utf-8').read())
root = os.path.dirname(DATA)
tr = os.path.join(root, 'train/images')
va = os.path.join(root, 'val/images')
print('train:', len(os.listdir(tr)) if os.path.isdir(tr) else 0)
print('val:', len(os.listdir(va)) if os.path.isdir(va) else 0)
assert os.path.isdir(tr) and len(os.listdir(tr)) > 0, 'train/images 비어 있음'

## 4.5 의심 프레임 전부 삭제 (선택)

파도 오탐처럼 **사람 수/물 구역 박스가 비정상적으로 많은** 프레임을 찾아 목록을 보여 준 뒤,  
아래 셀에서 `PURGE_ALL = True` 로 실행하면 **의심 파일을 전부 삭제**합니다.

- 학습 **전에** 실행하세요 (4절 직후).
- 진짜로 붐비는 정상 장면도 지워질 수 있으니, 먼저 `PURGE_ALL = False` 로 목록만 확인하세요.

In [ ]:
# 4.5) 의심 프레임 선별 + (옵션) 전부 삭제
# PURGE_ALL=False → 목록만 / True → train·val 에서 해당 jpg+txt 전부 삭제
import os, statistics
from pathlib import Path

PURGE_ALL = False   # ← 목록 확인 후 True 로 바꾸고 다시 실행

WATER_Y_TOP, WATER_Y_BOT = 0.45, 0.78
MIN_COUNT = 40      # 사람 과다 절대 하한
WATER_MIN = 25      # 물 구역 박스 과다(파도 오탐 전형)
Z = 3.0

# 4절에서 만든 DATA 경로 사용
root = Path(os.path.dirname(DATA)) if 'DATA' in dir() else Path('/content/gwangalli')
assert root.is_dir(), root

def read_counts(lbl_path: Path):
    total = water = 0
    try:
        for line in lbl_path.read_text(encoding='utf-8').splitlines():
            parts = line.split()
            if len(parts) < 5 or parts[0] != '0':
                continue
            cy = float(parts[2])
            total += 1
            if WATER_Y_TOP <= cy <= WATER_Y_BOT:
                water += 1
    except Exception:
        pass
    return total, water

# train/val (+ flat labels) 스캔
entries = []  # (stem, split, lbl, img)
for split in ('train', 'val', ''):
    lbl_dir = root / split / 'labels' if split else root / 'labels'
    img_dir = root / split / 'images' if split else root / 'images'
    if not lbl_dir.is_dir():
        continue
    for lp in sorted(lbl_dir.glob('*.txt')):
        entries.append((lp.stem, split, lp, img_dir / f'{lp.stem}.jpg'))

assert entries, f'라벨 없음: {root}'
counts = []
stats = {}
for stem, split, lp, ip in entries:
    t, w = read_counts(lp)
    stats[(stem, split)] = (t, w, lp, ip)
    counts.append(t)

med = statistics.median(counts)
mad = statistics.median([abs(c - med) for c in counts]) or 1.0
hi = max(MIN_COUNT, med + Z * 1.4826 * mad)
print(f'root={root}')
print(f'frames={len(counts)}  median={med:.0f}  MAD={mad:.1f}  high_thresh={hi:.0f}  water_min={WATER_MIN}')

flagged = []
for (stem, split), (t, w, lp, ip) in stats.items():
    reasons = []
    if t >= hi:
        reasons.append(f'many_persons({t})')
    if w >= WATER_MIN:
        reasons.append(f'many_water({w})')
    if reasons:
        flagged.append((stem, split, t, w, reasons, lp, ip))

flagged.sort(key=lambda x: -x[2])
print(f'의심 {len(flagged)}장')
for stem, split, t, w, reasons, *_ in flagged[:30]:
    print(f'  [{split or "flat"}] {stem}.jpg  {reasons}')
if len(flagged) > 30:
    print(f'  ... 외 {len(flagged) - 30}장')

if not PURGE_ALL:
    print('\\n→ 위 목록이 맞으면 PURGE_ALL = True 로 바꾼 뒤 이 셀을 다시 실행하세요.')
else:
    n_files = 0
    for stem, split, t, w, reasons, lp, ip in flagged:
        for p in (lp, ip):
            if p.exists():
                p.unlink()
                n_files += 1
    # 남은 장수
    left_tr = len(list((root / 'train' / 'images').glob('*.jpg'))) if (root / 'train' / 'images').is_dir() else 0
    left_va = len(list((root / 'val' / 'images').glob('*.jpg'))) if (root / 'val' / 'images').is_dir() else 0
    print(f'\\n삭제 완료: 의심 {len(flagged)}장 ({n_files} files)')
    print(f'남은 train={left_tr}  val={left_va}')
    print('이제 아래 5절 학습을 실행하세요.')

In [ ]:
# 5) 학습 — 속도 최우선 (RTX PRO 6000 96GB / A100 등)
# Python API only (!yolo 금지)
from ultralytics import YOLO
import os, torch

DATA = DATA if 'DATA' in dir() else '/content/gwangalli/data.yaml'
assert os.path.exists(DATA), DATA

# MAX_SPEED=True  → 벽시계 최소 (지금 GPU 권장)
# MAX_SPEED=False → 품질 우선 (imgsz 1024)
MAX_SPEED = True

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM≈{vram_gb:.0f}GB  MAX_SPEED={MAX_SPEED}')

if MAX_SPEED:
    IMGSZ = 768          # 1024보다 훨씬 빠름. 원거리↑면 960
    BATCH = -1           # AutoBatch: GPU 채워 처리량 최대
    WORKERS = min(16, os.cpu_count() or 8)
    CACHE = 'ram'        # epoch 2부터 I/O 병목 제거 (시스템 RAM 부족 시 False)
    EPOCHS, PATIENCE = 80, 15
else:
    IMGSZ, BATCH, WORKERS, CACHE = 1024, 16, 8, False
    EPOCHS, PATIENCE = 100, 30

BASE = 'yolo26s.pt'
try:
    model = YOLO(BASE)
except Exception as e:
    print('yolo26s 실패 → yolov8s:', e)
    BASE = 'yolov8s.pt'
    model = YOLO(BASE)

print('base:', BASE, '| imgsz', IMGSZ, '| batch', BATCH, '| workers', WORKERS, '| cache', CACHE)

train_kw = dict(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=WORKERS,
    cache=CACHE,
    amp=True,
    patience=PATIENCE,
    close_mosaic=5,
    plots=False,
    hsv_v=0.5,
    degrees=0.0,
    translate=0.05,
    scale=0.3,
    fliplr=0.5,
    project='/content/runs',
    name='gwangalli',
    exist_ok=True,
)
try:
    results = model.train(**train_kw)
except Exception as e:
    err = str(e).lower()
    if CACHE and ('memory' in err or 'ram' in err):
        print('cache=ram 실패 → cache=False, batch=48 재시도')
        train_kw.update(cache=False, batch=48 if BATCH == -1 else BATCH)
        results = model.train(**train_kw)
    else:
        raise

print('DONE best=', '/content/runs/gwangalli/weights/best.pt')
print('끝나면: 런타임 → 런타임 연결 해제 (CU 절약)')


In [ ]:
# 6) 검증 (학습과 같은 imgsz)
from ultralytics import YOLO
BEST = '/content/runs/gwangalli/weights/best.pt'
_imgsz = IMGSZ if 'IMGSZ' in dir() else 768
model = YOLO(BEST)
metrics = model.val(data=DATA, imgsz=_imgsz, device=0, batch=16, workers=8)
print('imgsz:', _imgsz)
print('mAP50:', round(float(metrics.box.map50), 4))
print('mAP50-95:', round(float(metrics.box.map), 4))
print('P:', round(float(metrics.box.mp), 4), 'R:', round(float(metrics.box.mr), 4))


In [ ]:
# 7) Drive 저장 + 브라우저 다운로드
import shutil, os
from google.colab import files

BEST = '/content/runs/gwangalli/weights/best.pt'
os.makedirs('/content/drive/MyDrive/gwangalli', exist_ok=True)
out = '/content/drive/MyDrive/gwangalli/yolo26s_beach_ft.pt'
shutil.copy(BEST, out)
print('Drive:', out, os.path.getsize(out)//1024, 'KB')
files.download(BEST)
print('로컬에 저장: vision/models/yolo26s_beach_ft.pt 후 서버 재시작')